# Sketch2Shape AI — TU-Berlin sketch classifier

Run this notebook on Kaggle with GPU and Internet enabled. It trains the project-owned sketch recognition model. It does **not** train a universal 3D generator.

In [ ]:
import os, random, json, time
from pathlib import Path
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path('/kaggle/working/sketch2shape_tuberlin')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Kaggle normally includes Torch/TorchVision. Install only the dataset helper.
!pip -q install 'datasets>=2.20,<4' 'accelerate>=0.30,<2'

In [ ]:
from datasets import load_dataset

DATASET_ID = 'sdiaeyu6n/tu-berlin'
dataset = load_dataset(DATASET_ID)
print(dataset)
print('Columns:', dataset[list(dataset.keys())[0]].column_names)
print('Example:', dataset[list(dataset.keys())[0]][0])

In [ ]:
# Adapt to the dataset's image/label columns without hard-coding its schema.
from datasets import ClassLabel

train_split = 'train' if 'train' in dataset else list(dataset.keys())[0]
eval_split = 'validation' if 'validation' in dataset else ('test' if 'test' in dataset else train_split)
train_raw, eval_raw = dataset[train_split], dataset[eval_split]

image_col = next(name for name, feature in train_raw.features.items() if feature.__class__.__name__.lower() == 'image')
label_col = next((name for name, feature in train_raw.features.items() if isinstance(feature, ClassLabel)), None)
if label_col is None:
    label_col = next(name for name in train_raw.column_names if name.lower() in {'label', 'category', 'class'})

label_feature = train_raw.features[label_col]
if not isinstance(label_feature, ClassLabel):
    labels = sorted(set(train_raw[label_col]))
    label_to_id = {label: i for i, label in enumerate(labels)}
    def encode_label(row):
        row['target'] = label_to_id[row[label_col]]
        return row
    train_raw, eval_raw = train_raw.map(encode_label), eval_raw.map(encode_label)
else:
    labels = list(label_feature.names)
    train_raw = train_raw.rename_column(label_col, 'target')
    eval_raw = eval_raw.rename_column(label_col, 'target')

num_classes = len(labels)
print({'image_column': image_col, 'classes': num_classes, 'first_labels': labels[:10]})
(OUTPUT_DIR / 'labels.json').write_text(json.dumps(labels, indent=2))

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms, models
from PIL import ImageOps

transform = transforms.Compose([
    transforms.Lambda(lambda image: image.convert('L').convert('RGB')),
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=12, translate=(0.06, 0.06), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Lambda(lambda image: image.convert('L').convert('RGB')),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SketchDataset(torch.utils.data.Dataset):
    def __init__(self, source, transform): self.source, self.transform = source, transform
    def __len__(self): return len(self.source)
    def __getitem__(self, index):
        row = self.source[index]
        return self.transform(row[image_col]), int(row['target'])

train_loader = DataLoader(SketchDataset(train_raw, transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
eval_loader = DataLoader(SketchDataset(eval_raw, eval_transform), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
# Three epochs is a Kaggle-friendly first training run. Increase after the pipeline is verified.
EPOCHS = 3
best_accuracy = 0.0

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), targets)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * targets.size(0)

    model.eval(); correct = total = 0
    with torch.no_grad():
        for images, targets in eval_loader:
            prediction = model(images.to(DEVICE, non_blocking=True)).argmax(1).cpu()
            correct += (prediction == targets).sum().item(); total += targets.size(0)
    accuracy = correct / max(1, total)
    print(f'Epoch {epoch + 1}/{EPOCHS} | loss={running_loss/len(train_raw):.4f} | val_acc={accuracy:.4f}')
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save({'state_dict': model.state_dict(), 'labels': labels, 'accuracy': accuracy}, OUTPUT_DIR / 'tuberlin_resnet18.pt')

print('Saved:', OUTPUT_DIR / 'tuberlin_resnet18.pt', 'best accuracy:', best_accuracy)

In [ ]:
# Prediction helper for the later sketch-to-reference worker.
def predict_sketch(pil_image, top_k=3):
    model.eval()
    with torch.no_grad():
        probabilities = model(eval_transform(pil_image).unsqueeze(0).to(DEVICE)).softmax(1)[0]
    values, indices = probabilities.topk(top_k)
    return [{'label': labels[index], 'confidence': round(float(value), 4)} for value, index in zip(values.cpu(), indices.cpu())]

example = eval_raw[0][image_col]
display(example)
print(predict_sketch(example))